# The DAGScheduler: Converting DAGs into Execution Stages

**Module**: 04_execution_engine  
**Notebook**: 03_dag_scheduler.ipynb  
**Environment**: Apache Spark 4.2.1 · Standalone Cluster · PySpark · Python 3.13 · Java 21  

---

## Learning Objectives

- **What will I learn?**: How the DAGScheduler decomposes physical plans into stages, handles ShuffleMapStages vs ResultStages, and manages stage retries.
- **Why does this topic exist?**: The DAGScheduler is the brain that manages distributed execution pipelines.
- **What problem does it solve?**: Understanding fetch failures and stage re-computations.
- **Where is it used in real-world Data Engineering?**: Diagnosing pipeline failures in production.

---

<!-- curriculum-enrichment:v1 -->

## Learning Contract

| Attribute | Value |
|---|---|
| **Priority** | Advanced |
| **Estimated effort** | 2–3 h |
| **Prerequisites** | DAG/plans, jobs/stages/tasks |
| **Required depth** | Deep internals: can explain stage creation and recovery |
| **Data Engineer** | Use stage information to debug jobs and understand retries. |
| **Senior Data Engineer** | Diagnose shuffle-stage failures and recomputation. |
| **Spark specialist** | Explain DAGScheduler event handling, stage submission, and task set management. |
| **Next topic** | `04_execution_engine/04_task_scheduler.ipynb` |

> **How to use this notebook**: read the problem first, then run the code, then connect the
> mechanism to the real-world cases and production notes. The goal is not to memorize API
> syntax; it is to be able to explain and defend a design decision.

---

## Problem and Motivation

### Why this concept exists

The DAGScheduler is the bridge between the physical plan and the task scheduler. It decides where stages
begin and end, submits stage attempts, and handles failures. Most "why did my job recompute?" questions are
answered by understanding which shuffle map outputs were lost and which stages had to be resubmitted.

### What happens if you ignore it

A team can still write code that runs on small data. The failure appears later: a job that
looks correct becomes slow, unstable, or expensive at scale, and the team lacks the vocabulary
to diagnose which Spark mechanism is responsible. This notebook builds that vocabulary and
ties it to observable behavior in the lab.

### A small mental model

Think of the topic as a contract between the logical work you describe and the physical work
Spark performs. The contract has a cost, a failure mode, and a set of trade-offs. The sections
below make each of those explicit.

---

## Core Concepts

### Definitions

| Term | Plain-language meaning | Technical meaning |
|---|---|---|
| **DAG** | Directed acyclic graph of stages | Built from RDD dependencies |
| **ShuffleMapStage** | Produces shuffle map outputs | One per shuffle boundary |
| **ResultStage** | Produces the final result | The stage that satisfies the action |
| **Stage attempt** | One execution of a stage | Can be retried after failure |
| **TaskSet** | Tasks for one stage | One task per partition |
| **MapOutputTracker** | Tracks shuffle map outputs | Driver-side metadata |

### How it works, step by step

1. An action calls `DAGScheduler.runJob()`.
2. The scheduler walks backward from the final RDD.
3. Shuffle dependencies define stage boundaries.
4. Missing parent stages are submitted first.
5. Each stage becomes a TaskSet and is offered to the TaskScheduler.
6. On failure, the scheduler may resubmit a stage or recompute lost map outputs.

### What to keep in mind

- The mechanism is not free; it consumes CPU, memory, network, or disk.
- The right choice depends on data size, distribution, latency target, and failure tolerance.
- A plan or UI observation is stronger evidence than an intuition about what "should" happen.

## 2. DAGScheduler Architecture

The `DAGScheduler` works backward from the final action target, identifying `ShuffleDependencies` to draw stage boundaries. Stages with no dependencies run first.

### 1. Initialize SparkSession on Standalone Cluster

We connect to `spark://spark-master:7077` with the driver configured to bind to container hostname `jupyter`.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import time

spark = (
    SparkSession.builder
    .appName("DAGScheduler_Lab")
    .master("spark://spark-master:7077")
    .config("spark.driver.host", "jupyter")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.eventLog.enabled", "true")
    .config("spark.eventLog.dir", "file:///opt/spark-events")
    .config("spark.sql.warehouse.dir", "file:///opt/spark-data/warehouse")
    .config("spark.hadoop.fs.permissions.umask-mode", "000")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print(f" SparkSession initialized successfully!")
print(f"Spark Version:      {spark.version}")
print(f"Master URL:         {spark.sparkContext.master}")
print(f"App Name:           {spark.sparkContext.appName}")
print(f"Default Parallelism: {spark.sparkContext.defaultParallelism}")
print(f"Spark UI available at: http://localhost:4040")

## 3. Practical: ShuffleMapStage vs ResultStage

In [ ]:
orders = spark.read.option("header", True).csv("/opt/spark-data/retail/orders.csv")
res = orders.groupBy("status").count()
print("Trigger action to observe stages in Spark UI:")
res.show()
print("Visit http://localhost:4040/stages/ to see ShuffleMapStage (Stage 0) and ResultStage (Stage 1)!")

---

## Experiment: Stage Retries on Fetch Failure

- **1. What are we testing?**: Explain DAGScheduler retry policy.
- **2. What do we expect?**: Upstream stage is retried if shuffle files are lost.

In [ ]:
print('Max stage failures:', spark.conf.get('spark.stage.maxConsecutiveAttempts', '4'))

**3. What happened?**: Default 4 attempts before aborting application.  
**4. Why did it happen?**: Transient network drops can cause fetch failures.  
**5. Practical Engineering Lesson**: DAGScheduler retries stages automatically upon transient fetch errors.

---

## Real-World Use Cases

> The scenarios below are illustrative. Company names are omitted, and any numbers are hypothetical unless they are measured in this lab.

### Use Case 1: Debugging a shuffle-stage failure

- **Industry**: Telecommunications
- **Scenario**: A nightly call-record aggregation fails after a worker restart.
- **Problem**: The stage retries but the job still fails, and the logs show missing shuffle outputs.
- **Why this concept is relevant**: The DAGScheduler must recompute the map stage that produced the lost shuffle blocks.
- **Implementation idea**: Inspect the Stages tab for failed stage attempts and missing map outputs; check worker disk and executor logs.
- **Expected result**: A clear diagnosis: recompute the map stage or fix the failing worker/storage.
- **Alternative**: A replicated shuffle service.
- **Trade-offs**: Spark's shuffle service can recover blocks on some cluster managers; the standalone lab has fewer guarantees.

### Use Case 2: Understanding job recovery

- **Industry**: E-commerce
- **Scenario**: An executor dies mid-job and the application continues.
- **Problem**: The team wants to know whether the final result is still correct and how much work was repeated.
- **Why this concept is relevant**: The DAGScheduler tracks stage attempts and resubmits only the necessary work.
- **Implementation idea**: Compare stage attempts, task retries, and shuffle read/write before and after the failure.
- **Expected result**: A correct result with a measurable recomputation cost.
- **Alternative**: A checkpointed/replicated pipeline.
- **Trade-offs**: Lineage recovery is automatic; checkpointing can reduce recomputation for long lineages.

---

## Business Value

- Improves incident diagnosis for shuffle and executor failures.
- Helps teams understand the real cost of retries.
- Supports better checkpoint and shuffle-service decisions.
- Reduces blind cluster restarts.

> **Note on numbers**: this repository does not claim production benchmarks. If you want to
> quantify a business case, measure the job in your own environment (runtime, shuffle bytes,
> spill, infrastructure cost) before and after the change.

---

## Production Engineering

### What changes when this runs in production

- Monitor stage attempts and failed tasks, not just job status.
- Keep shuffle data on reliable local disks; monitor disk pressure.
- Use checkpointing for long lineages and iterative jobs.
- Understand that a lost map output can force recomputation of a whole stage.
- Do not assume a successful retry means the underlying instability is fixed.

### Investigation checklist

- [ ] Open the Spark UI and identify the relevant tab for this topic.
- [ ] Compare the metric before and after a configuration or code change.
- [ ] Check driver and executor logs for warnings or retries.
- [ ] Confirm that the fix is measured, not assumed.

### Common production mistakes

1. Copying a local-lab setting into a production cluster without sizing it.
2. Optimizing a single job without checking the downstream SLA and data-quality impact.
3. Treating a successful run as proof of correctness; verify row counts, keys, and invariants.
4. Forgetting that retries and backfills must be idempotent.

### Experiment: Stage attempts after an executor failure

**Steps**

1. Run a multi-stage aggregation.
2. Stop one executor while the job is running (in a disposable lab).
3. Inspect stage attempts, failed tasks, and recomputed shuffle data.

**What to observe**: The scheduler resubmits the necessary stage/tasks; the job may still succeed.

**Healthy result**: The job completes with a measurable retry/recompute cost.

**Problematic result**: Repeated stage failures usually point to a persistent node or disk issue.

**Engineering takeaway**: Retries are a safety net, not a capacity plan.

---

## Trade-offs and Alternatives

### Advantages

- Automatic stage recovery
- Fine-grained resubmission
- Clear stage-level observability

### Limitations

- Recomputation can be expensive
- Shuffle loss can cascade
- Internal behavior is complex

### When to use it

- Debugging stage failures
- Fault-tolerance design
- Shuffle tuning

### When not to use it

- Relying on retries instead of fixing unstable nodes
- Ignoring repeated stage attempts

### Alternatives inside Spark

- TaskScheduler
- MapOutputTracker
- Checkpointing

### Alternatives outside Spark

- Replicated shuffle services
- Managed Spark recovery

### Performance implications

Every choice has a cost. Measure runtime, shuffle read/write, spill, GC time, and task
duration percentiles before and after the change. A local four-core cluster is useful for
observing mechanics, but it is not a production benchmark.

---

## Practical Exercises

### Exercise 1 (Beginner): Identify Stages
**Problem Statement**: Count stages in a simple select query.  
**Input / Context**: orders.select('order_id')

In [ ]:
# Sol 1

### Exercise 2 (Intermediate): Two Shuffles Stage Count
**Problem Statement**: Count stages in groupBy + repartition.  
**Input / Context**: Provided dataset

In [ ]:
# Sol 2

### Exercise 3 (Advanced): DAGScheduler Metrics
**Problem Statement**: Check stage duration in UI.  
**Input / Context**: Provided dataset

In [ ]:
# Sol 3

---

## Production Debugging Exercise

### Stage stuck at 99%

**Scenario**: A stage has one task running long after the others finish.

**Your task**: identify the most likely root cause, propose one diagnostic step, and propose
one fix. Write your answer before opening the model approach.

**Hints**

1. Compare task duration percentiles in the Stages tab.
2. Check for skew in the shuffle key.
3. Inspect spill and GC on the straggler executor.

<details>
<summary>Model approach (open after trying)</summary>

1. Reproduce the issue with the smallest possible dataset.
2. Use the Spark UI and the physical plan to locate the stage or operator that dominates.
3. Change one variable at a time and re-measure.
4. Record the before/after metric so the fix is defensible.

The exact fix depends on the evidence. The important habit is to separate allocation,
scheduling, shuffle, memory, and data-quality causes instead of guessing.

</details>

---

## Reference Solutions

<details>
<summary><b>Click to expand full solutions</b></summary>

#### Solution 1 (Identify Stages):
```python
orders.select('order_id').count() # 1 stage
```

#### Solution 2 (Two Shuffles Stage Count):
```python
orders.groupBy('status').count().repartition(4).count() # 3 stages
```

#### Solution 3 (DAGScheduler Metrics):
```python
print('Inspected UI Stages tab')
```

</details>

---

## Module Summary & Interview Guide

### Key Takeaways
- **DAGScheduler**: Splits DAG at shuffles into stages.

### Essential Technical Terms
- `ShuffleMapStage`: Writes shuffle files.
- `ResultStage`: Computes final action.

### Common Engineering Mistakes
- **Assuming 1 job = 1 stage**: Jobs have multiple stages if shuffles exist.

### Interview Questions on This Topic
- **Q: What creates a stage boundary?**
  - *A*: A wide dependency (ShuffleDependency).


### More Interview Questions

- **Q: What does the DAGScheduler do?**
  - *A*: It converts the RDD/plan DAG into stages at shuffle boundaries, submits stages, and handles stage/task failures and recomputation.
- **Q: What is a ShuffleMapStage?**
  - *A*: A stage that writes shuffle map outputs for a downstream stage; it is created at a wide dependency.

### What to Study Next
Open 04_task_scheduler.ipynb

---

### Tear Down SparkSession
Always close your session when finishing to release executor threads and driver ports.

In [ ]:
spark.stop()
print('SparkSession successfully stopped.')